In [1]:
import numpy as np
import matplotlib.pyplot as plt

def find_minimum_a_prime(g_values, g_hat_values):
    """
    Find minimum a' ∈ [0, 1) that satisfies:
    (1 - a')(ĝ_j^t)^2 ≤ (g_j^t)^2 ≤ 1/(1 - a')(ĝ_j^t)^2
    
    Parameters:
    g_values: array of g_j^t values
    g_hat_values: array of ĝ_j^t values (same length as g_values)
    
    Returns:
    dict with minimum a' and constraint analysis
    """
    
    # Convert to numpy arrays
    g = np.array(g_values)
    g_hat = np.array(g_hat_values)
    
    # Handle edge cases
    if len(g) != len(g_hat):
        raise ValueError("g and g_hat must have same length")
    
    # Remove pairs where g_hat = 0 (would cause division by zero)
    valid_mask = g_hat != 0
    g_valid = g[valid_mask]
    g_hat_valid = g_hat[valid_mask]
    
    if len(g_valid) == 0:
        return {
            'min_a_prime': 0.0,
            'feasible': True,
            'message': 'All g_hat values are zero, any a\' ∈ [0,1) works'
        }
    
    # Split into two inequalities:
    # Left:  (1 - a')(ĝ^t)^2 ≤ (g^t)^2  =>  a' ≥ 1 - (g^t)^2/(ĝ^t)^2
    # Right: (g^t)^2 ≤ 1/(1 - a')(ĝ^t)^2  =>  a' ≤ 1 - (ĝ^t)^2/(g^t)^2
    
    # Compute ratios
    g_squared = g_valid**2
    g_hat_squared = g_hat_valid**2
    
    # Left constraint: a' ≥ 1 - g^2/ĝ^2
    left_constraints = 1 - g_squared / g_hat_squared
    min_a_from_left = np.max(left_constraints)  # Most restrictive lower bound
    
    # Right constraint: a' ≤ 1 - ĝ^2/g^2 (only when g ≠ 0)
    right_constraints = []
    for i in range(len(g_valid)):
        if g_valid[i] != 0:
            right_constraints.append(1 - g_hat_squared[i] / g_squared[i])
    
    if len(right_constraints) > 0:
        max_a_from_right = np.min(right_constraints)  # Most restrictive upper bound
    else:
        max_a_from_right = 1.0  # No upper bound constraint
    
    # Domain constraint: a' ∈ [0, 1)
    min_a_from_left = max(0, min_a_from_left)  # Can't be below 0
    max_a_from_right = min(0.999999, max_a_from_right)  # Can't be 1 or above
    
    # Check feasibility
    if min_a_from_left > max_a_from_right:
        return {
            'min_a_prime': None,
            'feasible': False,
            'message': f'No feasible a\': need a\' ≥ {min_a_from_left:.6f} and a\' ≤ {max_a_from_right:.6f}',
            'left_bound': min_a_from_left,
            'right_bound': max_a_from_right
        }
    
    # The minimum feasible a' is the left bound (since we want to minimize a')
    return {
        'min_a_prime': min_a_from_left,
        'feasible': True,
        'feasible_range': [min_a_from_left, max_a_from_right],
        'left_bound': min_a_from_left,
        'right_bound': max_a_from_right,
        'message': f'Minimum a\' = {min_a_from_left:.6f}, feasible range: [{min_a_from_left:.6f}, {max_a_from_right:.6f}]'
    }

def verify_solution(a_prime, g_values, g_hat_values):
    """
    Verify that the chosen a' satisfies the original inequality
    """
    g = np.array(g_values)
    g_hat = np.array(g_hat_values)
    
    # Check each constraint
    violations = []
    
    for i in range(len(g)):
        if g_hat[i] == 0:
            continue  # Skip undefined cases
            
        left_side = (1 - a_prime) * (g_hat[i]**2)
        middle = g[i]**2
        right_side = (1 / (1 - a_prime)) * (g_hat[i]**2)
        
        left_satisfied = left_side <= middle
        right_satisfied = middle <= right_side
        
        if not (left_satisfied and right_satisfied):
            violations.append({
                'index': i,
                'left_side': left_side,
                'middle': middle,
                'right_side': right_side,
                'left_ok': left_satisfied,
                'right_ok': right_satisfied
            })
    
    return {
        'valid': len(violations) == 0,
        'violations': violations,
        'num_violations': len(violations)
    }

def analyze_constraints_graphically(g_values, g_hat_values):
    """
    Show the constraint bounds graphically
    """
    result = find_minimum_a_prime(g_values, g_hat_values)
    
    if not result['feasible']:
        print("Problem is infeasible!")
        return
    
    # Plot the feasible region
    a_range = np.linspace(0, 0.99, 1000)
    
    plt.figure(figsize=(10, 6))
    
    # Show feasible region
    min_a = result['left_bound']
    max_a = result['right_bound']
    
    plt.axvspan(min_a, max_a, alpha=0.3, color='green', label='Feasible Region')
    plt.axvline(min_a, color='red', linestyle='--', linewidth=2, label=f'Minimum a\' = {min_a:.4f}')
    
    plt.xlim(0, 0.99)
    plt.xlabel('a\'')
    plt.ylabel('Feasibility')
    plt.title('Feasible Region for a\'')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Add text annotation
    plt.text(min_a + 0.01, 0.5, f'Min a\' = {min_a:.6f}', 
             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    plt.show()
    
    return result

In [5]:
# Example data - replace with your actual batch data
np.random.seed(42)

# Generate some example g and g_hat values
n_samples = 10
g_values = np.random.uniform(-1, 1, n_samples)
g_hat_values = np.random.uniform(0.1, 2, n_samples)  # Avoid g_hat = 0

print("Example batch data:")
print("g values:", g_values[:5], "...")
print("ĝ values:", g_hat_values[:5], "...")
print()

# Find minimum a'
result = find_minimum_a_prime(g_values, g_hat_values)

print("Results:")
print(f"Feasible: {result['feasible']}")
if result['feasible']:
    print(f"Minimum a': {result['min_a_prime']:.8f}")
    print(f"Feasible range: [{result['left_bound']:.6f}, {result['right_bound']:.6f}]")
    
    # Verify the solution
    verification = verify_solution(result['min_a_prime'], g_values, g_hat_values)
    print(f"Solution verified: {verification['valid']}")
    if not verification['valid']:
        print(f"Number of violations: {verification['num_violations']}")
else:
    print(f"Problem infeasible: {result['message']}")

# Show graphical analysis
analyze_constraints_graphically(g_values, g_hat_values)

Problem is infeasible!


In [3]:
# Show the constant values you mentioned
print("\nConstant constraint bounds (for this batch):")
g = np.array(g_values)
g_hat = np.array(g_hat_values)
valid_mask = g_hat != 0

if np.sum(valid_mask) > 0:
    left_bounds = 1 - (g[valid_mask]**2) / (g_hat[valid_mask]**2)
    print(f"Left constraints (a' ≥): {left_bounds}")
    print(f"Most restrictive left bound: {np.max(left_bounds):.6f}")
    
    right_bounds = []
    for i in range(len(g)):
        if g_hat[i] != 0 and g[i] != 0:
            right_bounds.append(1 - (g_hat[i]**2) / (g[i]**2))
    
    if len(right_bounds) > 0:
        print(f"Right constraints (a' ≤): {right_bounds}")
        print(f"Most restrictive right bound: {np.min(right_bounds):.6f}")


Constant constraint bounds (for this batch):
Left constraints (a' ≥): [-2.25349143  0.78472497  0.9238716   0.8463877  -1.38505216 -1.35356768
 -0.69904006  0.55434646  0.95175424  0.59428847]
Most restrictive left bound: 0.951754
Right constraints (a' ≤): [np.float64(0.6926378872578876), np.float64(-3.6452206813937647), np.float64(-12.135702510465435), np.float64(-5.50989532529367), np.float64(0.580721957400705), np.float64(0.5751131317364246), np.float64(0.4114323602321275), np.float64(-1.2438955400253335), np.float64(-19.72720819999946), np.float64(-1.4648054723124968)]
Most restrictive right bound: -19.727208
